# Chapter 1: Basic Prompt Structure

- [Lesson](#lesson)
- [Exercises](#exercises)
- [Example Playground](#example-playground)

## Setup

Run the cell below once to load `MODEL_NAME` from the IPython store and define
the `get_completion` helper used throughout the chapter.

In [ ]:
# Python's built-in regular expression library (used by the graders)
import re
import ollama

# Retrieve the MODEL_NAME variable saved in Chapter 0 (the "How-To" notebook).
%store -r MODEL_NAME

def get_completion(user_prompt: str, system_prompt=""):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})
    response = ollama.chat(
        model=MODEL_NAME,
        messages=messages,
        options={"temperature": 0.0, "num_predict": 2000},
    )
    return response["message"]["content"]

MODEL_NAME

---

## Lesson

When you talk to a chat model through Ollama, you call `ollama.chat(...)`. At a
minimum that call needs:

- **`model`** — the name of a model you've pulled locally (e.g. `"granite4.1:3b"`).

- **`messages`** — a list of message objects. Chat models are trained on
  alternating **`user`** and **`assistant`** turns. You supply the conversation
  so far, and the model generates the next `assistant` turn. Each message is a
  dictionary with two keys:
  - **`role`** — one of `"system"`, `"user"`, or `"assistant"`.
  - **`content`** — the text of that turn.

  The first non-system message should be a `user` turn, and `user`/`assistant`
  turns should alternate after that.

You'll also commonly pass an **`options`** dictionary for generation settings:

- **`temperature`** — how much randomness is in the output. We set it to `0`
  for these lessons so results are as repeatable as possible.

- **`num_predict`** — the maximum number of tokens to generate. This is a hard
  cap, so a too-small value can cut the model off mid-sentence.

Finally, there's the **system prompt** (the `system` role), which we'll cover at
the end of this lesson.

### Examples

Let's see how the model responds to some well-formed prompts. Run each cell and the response will appear beneath it.

In [ ]:
# Prompt
USER_PROMPT = "What is the largest planet in our solar system?"

# Print the model's response
print(get_completion(USER_PROMPT))

In [ ]:
USER_PROMPT = "Give me a one-sentence pitch for a cozy board game about beekeeping."
print(get_completion(USER_PROMPT))

In [ ]:
USER_PROMPT = "How many strings does a standard violin have?"
print(get_completion(USER_PROMPT))

Now let's look at some prompts that **break the message format**.

First, a message that isn't a proper `{"role": ..., "content": ...}` dictionary
at all — here it's just a bare set containing a string. This raises a Python /
client error before anything ever reaches the model.

In [ ]:
# This message is a set literal, not a dict with "role" and "content" keys.
# Running this raises an error.
response = ollama.chat(
    model=MODEL_NAME,
    messages=[
        {"Tell me a joke about penguins."}
    ],
    options={"temperature": 0.0, "num_predict": 2000},
)

print(response["message"]["content"])

Next, a conversation that doesn't alternate roles — two `user` turns back to
back with no `assistant` turn between them.

A note specific to Ollama: many hosted chat APIs (e.g. OpenAi, Anthropic, Gemini) will
**reject** this outright, because they strictly enforce alternating turns.
Ollama is permissive and will just run it. But non-alternating
messages are still off-distribution from how the model was trained, so you
should keep turns alternating anyway. The cell below still runs — notice
that the model has to guess how to stitch two questions together.

In [ ]:
# Two user turns in a row, with no assistant turn between them.
response = ollama.chat(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "What is the boiling point of water at sea level?"},
        {"role": "user", "content": "And how does that change at high altitude?"}
    ],
    options={"temperature": 0.0, "num_predict": 2000},
)

print(response["message"]["content"])

The rule of thumb: **`user` and `assistant` turns should alternate, and the
conversation should start with a `user` turn** (optionally preceded by a single
`system` message). You can include several `user`/`assistant` pairs to simulate
a multi-turn conversation, and you can even put words in a final `assistant`
turn for the model to continue from (called prefilling) — more on that in a later chapter.

#### System Prompts

A **system prompt** lets you give the model context, instructions, and
guidelines *before* it sees the user's question. Structurally it sits apart from
the back-and-forth `user`/`assistant` turns: in our helper it's the optional
`system_prompt` argument, which gets added as a leading `{"role": "system", ...}`
message. Leave it as an empty string when you don't need one.

#### System Prompt Example

In [ ]:
# System prompt
SYSTEM_PROMPT = ""

# User prompt
USER_PROMPT = "What do I need to plan a birthday party?"

# Print the model's response
print(get_completion(USER_PROMPT, SYSTEM_PROMPT))

In [ ]:
# System prompt
SYSTEM_PROMPT = "You are a terse mission-control operator. Respond in clipped, all-caps radio transmissions, and end every message with the word OVER."

# User prompt
USER_PROMPT = "What do I need to plan a birthday party?"

# Print the model's response
print(get_completion(USER_PROMPT, SYSTEM_PROMPT))

Notice how the **same question** produces a completely different style of
answer once the system prompt is in place. A well-written system prompt is one
of the most reliable ways to steer tone, format, and how closely the model
follows your rules.

Now for some exercises. If you'd rather experiment freely first, jump to the
[**Example Playground**](#example-playground) at the bottom.

---

## Exercises
- [Exercise 1.1 - Primary Colors](#exercise-11---primary-colors)
- [Exercise 1.2 - Beep Boop](#exercise-12---beep-boop)

### Exercise 1.1 - Primary Colors

Edit the `USER_PROMPT` below so the model names the **three primary colors**
(red, yellow, and blue). The grader checks that all three appear in the
response.

In [ ]:
# User prompt - this is the only field you should change
USER_PROMPT = "[REPLACE THIS TEXT]"

# Get the model's response
response = get_completion(USER_PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return all(color in text.lower() for color in ["red", "yellow", "blue"])

# Print the response and the corresponding grade
print(response)
print("\n--------------------------- GRADING ---------------------------")
print("This exercise has been correctly solved:", "✅" if grade_exercise(response) else "❌")

❓ If you want a hint, run the cell below!

In [ ]:
from hints import exercise_1_1_hint; print(exercise_1_1_hint)

### Exercise 1.2 - Spanish Greeting

- Modify the `SYSTEM_PROMPT` so the model answers **in Spanish**.
- Change *only* the `SYSTEM_PROMPT` — leave the `PROMPT` alone.

In [ ]:
# System prompt - this is the only field you should change
SYSTEM_PROMPT = "[REPLACE THIS TEXT]"

# User prompt
USER_PROMPT = "Hello, how are you?"

# Get the model's response
response = get_completion(USER_PROMPT, SYSTEM_PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search(r"hola", text.lower()))

# Print the response and the corresponding grade
print(response)
print("\n--------------------------- GRADING ---------------------------")
print("This exercise has been correctly solved:", "✅" if grade_exercise(response) else "❌")

❓ If you want a hint, run the cell below!

In [ ]:
from hints import exercise_1_2_hint; print(exercise_1_2_hint)

### Congrats!

If both exercises pass, you've got the basics of message structure and system
prompts down. On to the next chapter — happy prompting!

---

## Example Playground

A space to experiment freely with the prompts from this lesson and see how
changes affect the model's responses.

In [ ]:
USER_PROMPT = "What is the largest planet in our solar system?"
print(get_completion(USER_PROMPT))

In [ ]:
USER_PROMPT = "Give me a one-sentence pitch for a cozy board game about beekeeping."
print(get_completion(USER_PROMPT))

In [ ]:
SYSTEM_PROMPT = "You are a terse mission-control operator. Respond in clipped, all-caps radio transmissions, and end every message with the word OVER."
USER_PROMPT = "What do I need to plan a birthday party?"
print(get_completion(USER_PROMPT, SYSTEM_PROMPT))